# RPS notebook

1. get data
* 

2. training

In [65]:
# imports
import os
import csv
import pandas as pd
# import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.io import decode_image
from torchvision.transforms import v2

import torch.optim as optim

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


## Data

* Get data, restructure

* dataloader class

In [66]:
# Data in file structure

if not os.path.isdir("rps_data_sample") :
    !wget https://storage.googleapis.com/mediapipe-tasks/gesture_recognizer/rps_data_sample.zip
    !unzip rps_data_sample.zip
    !rm rps_data_sample.zip
    with open("labels.csv", "w") as labels_csv :
        writer = csv.writer(labels_csv)
        for i, d in enumerate(os.listdir("rps_data_sample")) :
            for f in os.listdir(f"rps_data_sample/{d}") :
                writer.writerow([f"{d}/{f}", i])

Labels = ["Rock", "Scissors", "None", "Paper"]


In [67]:
class RPS_Dataset(Dataset) :
    def __init__(self, labels_file_path : str, data_path : str, transform : v2.Compose | None = None) -> None:
        self._entries = pd.read_csv(labels_file_path, header = None)
        self._dataDir = data_path
        self._transform = transform

    def __len__(self) -> int:
        return len(self._entries)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int]:
        item_path = os.path.join(self._dataDir, str(self._entries.iloc[index, 0]))
        item = decode_image(item_path)
        if self._transform :
            item = self._transform(item)
        label = self._entries.iloc[index, 1]
        return item, label
    
    def set_transform(self, transform : v2.Compose) :
        self._transform = transform
    
#Transforms
transforms = v2.Compose([
    v2.CenterCrop([600, 600]), # Pads (with 0) small ims and crops large ims 
    #Normalization
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transforms_train = v2.Compose([
    v2.RandomCrop([64, 64]),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transforms_test = v2.Compose([
    v2.Resize([224,224]),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
    
dataset = RPS_Dataset(labels_file_path="labels.csv", data_path="rps_data_sample", transform= transforms)
# dataset = dataset.to(device)
train, val = random_split(dataset, [0.8, 0.2])

rps_train_Loader = DataLoader(train, batch_size=64, shuffle=True)
rps_val_Loader = DataLoader(val, batch_size=64, shuffle=True)

In [68]:
# import matplotlib.pyplot as plt

# datset1 = RPS_Dataset(labels_file_path="labels.csv", data_path="rps_data_sample")

# # Check widths and heights of images
# xs, ys = ([], [])
# for d in datset1 : 
#     x = d[0].shape[1]
#     y = d[0].shape[2]
#     xs.append(x)
#     ys.append(y)

# plt.scatter(xs,ys)
# plt.show()



## Training

## Simple Torch CNN Example 

[Torch CNN example](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)

(accuracy of about 70%)

In [69]:
# Torch CNN example

from torch import nn
import torch.nn.functional as F

class RPS_Classifier_torch(nn.Module): 
    def __init__(self, n_classes : int, channels : int = 3, dims : tuple[int, int] = (600, 600)) -> None:
        super().__init__()
        x = int(dims[0]/2/2 - 3)
        y = int(dims[1]/2/2 - 3)
        self.conv1 = nn.Conv2d(channels, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * x * y, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, n_classes)

    def forward(self, item : torch.Tensor): 
        x = self.pool(F.relu(self.conv1(item)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) # flatten all dimensions except batch
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [75]:
# define net + hyper params

net = RPS_Classifier_torch(4, dims=(600,600))
net = net.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


# Training Loop

for epoch in range(15):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(rps_train_Loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()

        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
    print(f'Epoch {epoch} loss: {running_loss :.3f}')

print('Finished Training')

Epoch 0 loss: 9.480
Epoch 1 loss: 8.338
Epoch 2 loss: 7.025
Epoch 3 loss: 6.480
Epoch 4 loss: 6.003
Epoch 5 loss: 5.555
Epoch 6 loss: 5.131
Epoch 7 loss: 5.133
Epoch 8 loss: 4.905
Epoch 9 loss: 4.680
Epoch 10 loss: 4.327
Epoch 11 loss: 3.897
Epoch 12 loss: 3.090
Epoch 13 loss: 2.532
Epoch 14 loss: 2.155
Finished Training


In [76]:
correct = 0
total = 0
# since we're not training, we don't need to calculate the gradients for our outputs
with torch.no_grad():
    for data in rps_val_Loader:
        images, labels = data[0].to(device), data[1].to(device)
        # calculate outputs by running images through the network
        outputs = net(images)
        # the class with the highest energy is what we choose as prediction
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the test images: {100 * correct // total} %')

Accuracy of the network on the test images: 69 %


## ConvNext

In [27]:
# pretrained Convnext

from torchvision import models

convnet = models.convnext_tiny(weights= models.ConvNeXt_Tiny_Weights.DEFAULT)
convnet

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 191MB/s] 


ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [30]:
a = dataset[3][0]
print(a)
convnet(a[None, :, :, :])

tensor([[[-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         ...,
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179],
         [-2.1179, -2.1179, -2.1179,  ..., -2.1179, -2.1179, -2.1179]],

        [[-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         ...,
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357],
         [-2.0357, -2.0357, -2.0357,  ..., -2.0357, -2.0357, -2.0357]],

        [[-1.8044, -1.8044, -1.8044,  ..., -1.8044, -1.8044, -1.8044],
         [-1.8044, -1.8044, -1.8044,  ..., -1

tensor([[-1.1974e+00, -4.7649e-03, -4.6290e-01, -6.3913e-01,  5.3455e-01,
          3.5914e-01, -3.6466e-01, -1.7472e-01, -4.1170e-01, -4.4749e-01,
         -5.1861e-01, -1.1160e+00, -8.1101e-01, -6.6244e-01, -1.3554e+00,
         -1.3738e+00, -1.2095e+00, -1.6529e+00, -6.2650e-01, -9.9480e-01,
         -9.4273e-01,  9.9207e-02,  1.6744e-01,  1.6748e-01, -8.0840e-01,
         -2.0411e-01,  1.0880e-01, -7.4656e-01, -1.5822e-01,  2.1828e-01,
         -4.5695e-01,  1.0207e+00,  4.1234e-01, -1.4882e-01,  1.1222e+00,
         -8.4326e-01, -2.5206e-01, -1.1957e+00,  8.8360e-01, -6.6481e-01,
          1.8157e-01,  1.3091e-01,  3.0406e-01, -2.7191e-01, -1.2109e-01,
          1.4712e-03,  1.5687e-01,  3.2460e-01, -5.4305e-01,  1.9802e-01,
         -8.4659e-01, -2.3036e-01,  6.6950e-01,  3.7165e-01,  4.2690e-01,
         -6.8940e-02, -7.8189e-01, -9.7313e-01, -5.2403e-01,  7.6546e-01,
          7.5705e-01, -2.6263e-01, -1.4671e-01,  6.5953e-01,  2.0137e-01,
         -3.2228e-01,  1.4785e+00, -1.